# Usar Jev (TypeSafe AI) vía OpenRouter, con el SDK oficial

Jev **no es un modelo de chat**: no se llama con `chat.completions.create`. TypeSafe publica un SDK propio (`typesafe_sdk`) que envuelve su endpoint de decisiones (`/api/alpha/decisions`) con un método, `system_one()`.

En vez de `messages` (system/user/assistant), la llamada lleva:
- **`state`**: el contexto — en nuestro caso, la telemetría del rover.
- **`questions`**: un diccionario de preguntas puntuales sobre ese estado, cada una tipada (`noul` = sí/no con probabilidad, `choice` = elegir una opción, `score` = puntaje en una escala ordenada).

La respuesta (`result.answers`) viene **ya estructurada** por pregunta — no hay que parsear texto libre.

**Antes de correr esto:**
1. `pip install typesafe_sdk requests python-dotenv`
2. Un archivo `.env` **en esta misma carpeta** (`.handoff/`) con:
   ```
   OPENROUTER_API_KEY=sk-or-...
   ```
   `.handoff/` ya está en `.gitignore`, así que el `.env` no se sube al repo.
3. El simulador corriendo en otra terminal: `python server.py` (ver `CLAUDE.md` del repo).

**Nota sobre `model`**: la documentación del SDK usa `"jev-1.13"` (sin el prefijo `typesafe/` que sí lleva la llamada HTTP cruda). Si el SDK tira error de modelo no encontrado, probar con `"typesafe/jev-1.13"`.

Referencias: https://openrouter.ai/docs/guides/community/typesafe-sdk · https://openrouter.ai/docs/guides/community/jev-tutorial

In [5]:
# !pip install typesafe_sdk requests python-dotenv

## 1 · El cliente

`TypeSafeClient` apunta a OpenRouter con `base_url="https://openrouter.ai/api"`. La key se carga del `.env` con `python-dotenv` — nunca escrita en una celda.

In [ ]:
import os
import requests
from pathlib import Path
from dotenv import load_dotenv
from typesafe_sdk import TypeSafeClient

load_dotenv(Path.cwd() / ".env")

API_KEY = os.environ.get("OPENROUTER_API_KEY")

jev = TypeSafeClient(
    api_key=API_KEY,
    base_url="https://openrouter.ai/api",
)


## 2 · Ejemplo mínimo

Una pregunta `choice` suelta, para confirmar que la key y el modelo funcionan antes de meter telemetría real.

In [ ]:
result = jev.system_one(
    model="jev-1.13",
    state={"mensaje": "hola"},
    questions={
        "saludo": {
            "type": "choice",
            "instructions": "Que tipo de mensaje es este?",
            "criteria": {
                "saludo": "Es un saludo simple.",
                "pregunta": "Es una pregunta.",
                "otro": "Ninguna de las anteriores.",
            },
        }
    },
)

print(result.answers)

{'saludo': ChoiceAnswer(type='choice', choice='saludo', confidence=1.0, probabilities={'pregunta': 0.0, 'otro': 0.0, 'saludo': 1.0})}


## 3 · Las tres piezas de una llamada

- **`state`**: el contexto — acá, la telemetría real del rover (misma forma que `GET /api/telemetry`).
- **`questions`**: qué le preguntamos sobre ese estado. Para pilotar el rover usamos una sola pregunta `choice` llamada `action`, con las 5 acciones posibles como opciones — así Jev elige entre un conjunto cerrado, nunca inventa una acción que no existe.
- **`result.answers`**: la respuesta. Para una `choice` viene como `{"type": "choice", "choice": "forward", "confidence": 0.82, "probabilities": {...}}`.

**Sobre `value`** (metros o grados): el endpoint de decisiones no genera números libres — las preguntas `score` dan un puntaje en una escala *ordenada y acotada* (vos definís las etiquetas), no un float cualquiera. Por eso, para esta clase, usamos **pasos fijos**: cada acción mueve siempre la misma cantidad (1 m para forward/backward, 30° para los giros). Es la misma idea que ya usa `client_test.py`. Si más adelante quieren que Jev también elija la magnitud, se puede sumar una pregunta `score` separada (ver comentario al final de la sección 4).

In [ ]:
ACTION_QUESTION = {
    "action": {
        "type": "choice",
        "instructions": (
            "Dado el estado del rover, que debe hacer ahora mismo? "
            "Si obstacle_ahead es true, nunca elijas forward: gira o frena. "
            "Si overturned es true, elegi stop."
        ),
        "criteria": {
            "forward": "Avanzar un paso. Solo si no hay obstaculo cerca y no esta volcado.",
            "backward": "Retroceder un paso. Usar si esta muy pegado a un obstaculo.",
            "turn_left": "Girar a la izquierda un paso. Usar para esquivar por la izquierda.",
            "turn_right": "Girar a la derecha un paso. Usar para esquivar por la derecha.",
            "stop": "Detenerse. Usar si esta volcado o la situacion es insegura.",
        },
    }
}

# Ejemplo de telemetria real (misma forma que devuelve GET /api/telemetry)
#
# Situacion: el rover esta parado en (12.4, 3.1), mirando hacia heading=90
# (eje +Y). El sensor frontal (cono de 60 grados, alcance 4 m) detecto una
# roca a 2.1 m de distancia -> obstacle_ahead=True, obstacle_type="rock".
# Todavia no choco (blocked=False) ni esta volcado (overturned=False,
# pitch/roll casi en cero = terreno plano). Bateria en 87%, sin apuro.
#
# Respuesta correcta esperada: NUNCA "forward" (chocaria con la roca).
# Lo correcto es "turn_left" o "turn_right" para esquivarla (cualquiera de
# los dos vale, no hay info de hacia donde conviene ir globalmente todavia).
# "stop" seria demasiado conservador pero no incorrecto; "backward" no tiene
# sentido porque el rover ni siquiera esta pegado a la roca.
telemetria_ejemplo = {
    "x": 12.4, "y": 3.1, "z": 0.6,
    "heading": 90.0, "pitch": 1.2, "roll": -0.4,
    "speed": 1.0, "battery": 87.3,
    "obstacle_ahead": True, "obstacle_distance": 2.1, "obstacle_type": "rock",
    "blocked": False, "blocked_by": None, "overturned": False,
    "odometer": 14.8, "timestamp": 1733160000.0,
}

result = jev.system_one(model="jev-1.13", state=telemetria_ejemplo, questions=ACTION_QUESTION)
print(result.answers)

{'action': ChoiceAnswer(type='choice', choice='turn_left', confidence=0.31, probabilities={'stop': 0.04, 'backward': 0.06, 'forward': 0.0, 'turn_left': 0.46, 'turn_right': 0.44})}
